             MICROPHONE
                 ↓
                VAD
       "Is someone speaking?"
                 ↓
                ASR
        "What did they say?"
                 ↓
       SHOULD-RESPOND BERT
        "Should I answer?"
                 ↓
                LLM
        "What should I say?"
                 ↓
                TTS
        "Speak the answer"
                 ↓
              SPEAKER

             MICROPHONE
                  │
                  ▼
             Silero VAD
                  │
          ┌───────┴───────┐
          │               │
       silence          speech
          │               │
          │               ▼
          │          collect audio
          │               │
          │        short pause?
          │               │
          │          keep waiting
          │               │
          │       long silence
          │               │
          │               ▼
          │             Whisper
          │               │
          │               ▼
          │       Should AI Respond?
          │               │
          │          ┌────┴────┐
          │          │         │
          │          NO       YES
          │          │         │
          │          │         ▼
          │          │       LLM
          │          │         │
          │          │         ▼
          │          │        TTS
          │          │         │
          └──────────┴─────────┘

In [2]:
import torch
import sounddevice as sd
import numpy as np
import time

# Load Silero VAD
model, utils = torch.hub.load(
    repo_or_dir="snakers4/silero-vad",
    model="silero_vad",
    trust_repo=True
)

(get_speech_timestamps, _, read_audio, _, _) = utils

SAMPLE_RATE = 16000
RECORD_SECONDS = 10

print("Recording for 10 seconds...")
audio = sd.rec(
    int(RECORD_SECONDS * SAMPLE_RATE),
    samplerate=SAMPLE_RATE,
    channels=1,
    dtype="float32"
)
sd.wait()

audio = audio.squeeze()

# Convert numpy -> torch
audio_tensor = torch.from_numpy(audio)

speech_timestamps = get_speech_timestamps(
    audio_tensor,
    model,
    sampling_rate=SAMPLE_RATE
)

print("\nSpeech detected:")

if not speech_timestamps:
    print("No speech detected.")
else:
    for segment in speech_timestamps:
        start = segment["start"] / SAMPLE_RATE
        end = segment["end"] / SAMPLE_RATE
        print(f"{start:.2f}s → {end:.2f}s")

Using cache found in /home/keerthivardhan/.cache/torch/hub/snakers4_silero-vad_master
/home/keerthivardhan/Desktop/ProductionProjects/autonomus-assistent/.venv/lib/python3.12/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


Recording for 10 seconds...

Speech detected:
3.68s → 4.57s
5.25s → 6.33s


In [4]:
import sounddevice as sd
from scipy.io.wavfile import write
import subprocess
from pathlib import Path

SAMPLE_RATE = 16000
RECORD_SECONDS = 5

# Path to whisper.cpp
WHISPER_DIR = Path("../whisper.cpp")

MODEL = WHISPER_DIR / "models" / "ggml-base.en.bin"
WHISPER_CLI = WHISPER_DIR / "build" / "bin" / "whisper-cli"

AUDIO_FILE = Path("test_audio.wav")

print("Speak now...")

audio = sd.rec(
    int(RECORD_SECONDS * SAMPLE_RATE),
    samplerate=SAMPLE_RATE,
    channels=1,
    dtype="int16"
)

sd.wait()

print("Recording finished.")

write(AUDIO_FILE, SAMPLE_RATE, audio)

result = subprocess.run(
    [
        str(WHISPER_CLI),
        "-m", str(MODEL),
        "-f", str(AUDIO_FILE),
        "-nt"
    ],
    capture_output=True,
    text=True
)

print("\nWhisper output:")
print(result.stdout)

Speak now...
Recording finished.

Whisper output:

 I am going to put a test chest on my wrist


## Integrating VAD + ASR

In [12]:
import torch
import sounddevice as sd
import numpy as np
import time
from scipy.io.wavfile import write
import subprocess
from pathlib import Path

SAMPLE_RATE = 16000
RECORD_SECONDS = 5

# Path to whisper.cpp
WHISPER_DIR = Path("../whisper.cpp")

MODEL = WHISPER_DIR / "models" / "ggml-base.en.bin"
WHISPER_CLI = WHISPER_DIR / "build" / "bin" / "whisper-cli"

AUDIO_FILE = Path("test_audio.wav")
# Load Silero VAD
model, utils = torch.hub.load(
    repo_or_dir="snakers4/silero-vad",
    model="silero_vad",
    trust_repo=True
)

(get_speech_timestamps, _, read_audio, _, _) = utils

SAMPLE_RATE = 16000
RECORD_SECONDS = 10

print("Recording for 10 seconds...")
audio = sd.rec(
    int(RECORD_SECONDS * SAMPLE_RATE),
    samplerate=SAMPLE_RATE,
    channels=1,
    dtype="float32"
)
sd.wait()

audio = audio.squeeze()

# Convert numpy -> torch
audio_tensor = torch.from_numpy(audio)

speech_timestamps = get_speech_timestamps(
    audio_tensor,
    model,
    sampling_rate=SAMPLE_RATE
)

print("\nSpeech detected:")
PADDING_SECONDS = 0.3
if not speech_timestamps:
    print("No speech detected.")
else:    
    print("Recording finished.")
    
    for i,segment in enumerate(speech_timestamps):
        print("VAD output")

        padding_samples = int(PADDING_SECONDS * SAMPLE_RATE)

        
        start_sample = segment["start"]
        end_sample = segment["end"]

        start_sample = max(0, start_sample - padding_samples)
        end_sample = min(len(audio), end_sample + padding_samples)
    
        start_time = start_sample / SAMPLE_RATE
        end_time = end_sample / SAMPLE_RATE
        print(f"{i + 1}. {start_time:.2f}s → {end_time:.2f}s")


        speech_audio = audio[start_sample:end_sample]
        speech_file = Path(f"speech_{i + 1}.wav")
        write(
                speech_file,
                SAMPLE_RATE,
                speech_audio
            )
    
        # write(AUDIO_FILE, SAMPLE_RATE, audio)
        
        result = subprocess.run(
            [
                str(WHISPER_CLI),
                "-m", str(MODEL),
                "-f", str(speech_file),
                "-nt"
            ],
            capture_output=True,
            text=True
        )

        print("\nWhisper output:")
        print(result.stdout)

Using cache found in /home/keerthivardhan/.cache/torch/hub/snakers4_silero-vad_master


Recording for 10 seconds...

Speech detected:
Recording finished.
VAD output
1. 1.33s → 2.47s

Whisper output:

 Hi.
VAD output
2. 2.20s → 3.63s

Whisper output:

 Uh, hi.
VAD output
3. 3.13s → 6.57s

Whisper output:

 I have decreased from 100 seconds to 10 seconds.
VAD output
4. 6.26s → 8.97s

Whisper output:

 I'm the habadded bardin
VAD output
5. 8.73s → 10.00s

Whisper output:

 in the spot logic.


## end of turn detuction

In [14]:
import torch
import sounddevice as sd
import numpy as np
from scipy.io.wavfile import write
from pathlib import Path
import subprocess
import time

SAMPLE_RATE = 16000
CHUNK_SIZE = 512

SPEECH_THRESHOLD = 0.5
SILENCE_DURATION = 1.0
MAX_UTTERANCE_SECONDS = 30

WHISPER_DIR = Path("../whisper.cpp")
MODEL = WHISPER_DIR / "models" / "ggml-base.en.bin"
WHISPER_CLI = WHISPER_DIR / "build" / "bin" / "whisper-cli"

AUDIO_FILE = Path("utterance.wav")


# Reset Silero's internal state
model.reset_states()

print("Listening...")
print("Speak now. I will wait until you finish your turn.\n")

audio_chunks = []
speech_started = False
silence_start = None

start_time = time.time()

with sd.InputStream(
    samplerate=SAMPLE_RATE,
    channels=1,
    dtype="float32",
    blocksize=CHUNK_SIZE
) as stream:

    while True:

        # Read a small audio chunk
        chunk, overflowed = stream.read(CHUNK_SIZE)

        chunk = chunk[:, 0]

        # Convert numpy -> torch
        chunk_tensor = torch.from_numpy(chunk)

        # Ask Silero: is this speech?
        speech_probability = model(
            chunk_tensor,
            SAMPLE_RATE
        ).item()

        is_speech = speech_probability >= SPEECH_THRESHOLD

        # --------------------------------
        # Speech detected
        # --------------------------------

        if is_speech:

            if not speech_started:
                print("Speech detected...")
                speech_started = True

            audio_chunks.append(chunk.copy())

            # User started speaking again
            silence_start = None

        # --------------------------------
        # Silence detected
        # --------------------------------

        else:

            if speech_started:

                audio_chunks.append(chunk.copy())

                if silence_start is None:
                    silence_start = time.time()

                silence_time = time.time() - silence_start

                # User has been silent long enough
                if silence_time >= SILENCE_DURATION:
                    print("\nUser finished speaking.")
                    break

        # Safety timeout
        if time.time() - start_time >= MAX_UTTERANCE_SECONDS:
            print("\nMaximum utterance duration reached.")
            break


# --------------------------------
# Nothing was spoken
# --------------------------------

if not speech_started:

    print("No speech detected.")

else:

    # Combine all chunks
    audio = np.concatenate(audio_chunks)

    # Save audio
    write(
        AUDIO_FILE,
        SAMPLE_RATE,
        audio
    )

    print("Calling Whisper...")

    result = subprocess.run(
        [
            str(WHISPER_CLI),
            "-m", str(MODEL),
            "-f", str(AUDIO_FILE),
            "-nt"
        ],
        capture_output=True,
        text=True
    )

    print("\nWhisper output:")
    print(result.stdout.strip())

Listening...
Speak now. I will wait until you finish your turn.

Speech detected...

User finished speaking.
Calling Whisper...

Whisper output:
I have actually I am testing a new code cell which will wait until user faces is turned and only then this SAD module sends audio to whisper to trans drive.


In [17]:
from ollama import chat
LLM_MODEL = "qwen2.5-coder:3b"
response = chat(
        model=LLM_MODEL,
        messages=[
            {
                "role": "user",
                "content": "hi"
            }
        ]
    )

response

ChatResponse(model='qwen2.5-coder:3b', created_at='2026-09-06T13:19:01.04885094Z', done=True, done_reason='stop', total_duration=8439810268, load_duration=6033314554, prompt_eval_count=30, prompt_eval_duration=927326930, eval_count=22, eval_duration=1396361898, message=Message(role='assistant', content="Hello! How can I assist you today? Is there anything specific you'd like to know or discuss?", thinking=None, images=None, tool_name=None, tool_calls=None), logprobs=None)

In [18]:
import torch
import sounddevice as sd
import numpy as np
from scipy.io.wavfile import write
from pathlib import Path
import subprocess
import time
from transformers import AutoTokenizer
import onnxruntime as ort
from ollama import chat


# ============================================================
# Configuration
# ============================================================

SAMPLE_RATE = 16000
CHUNK_SIZE = 512

SPEECH_THRESHOLD = 0.5
SILENCE_DURATION = 1.0
MAX_UTTERANCE_SECONDS = 30

LLM_MODEL = "qwen2.5-coder:3b"

WHISPER_DIR = Path("../whisper.cpp")
WHISPER_MODEL = WHISPER_DIR / "models" / "ggml-base.en.bin"
WHISPER_CLI = WHISPER_DIR / "build" / "bin" / "whisper-cli"

AUDIO_FILE = Path("utterance.wav")

SHOULD_RESPOND_CLASS = 1


# ============================================================
# Load Should AI Respond model
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    "./should_ai_respond_model"
)

print("Loaded tokenizer")


int8_session = ort.InferenceSession(
    "./should_ai_respond_int8.onnx",
    providers=["CPUExecutionProvider"]
)

print("Loaded INT8 BERT classification model")


# ============================================================
# Helper
# ============================================================

def softmax(x):
    exp_x = np.exp(
        x - np.max(x, axis=1, keepdims=True)
    )
    return exp_x / exp_x.sum(
        axis=1,
        keepdims=True
    )


# ============================================================
# Main loop
# ============================================================

print("Listening...")
print("Speak now.\n")


while True:

    # Reset state for every utterance

    audio_chunks = []
    speech_started = False
    silence_start = None

    utterance_start_time = time.time()

    model.reset_states()

    with sd.InputStream(
        samplerate=SAMPLE_RATE,
        channels=1,
        dtype="float32",
        blocksize=CHUNK_SIZE
    ) as stream:

        while True:

            chunk, overflowed = stream.read(CHUNK_SIZE)

            chunk = chunk[:, 0]

            chunk_tensor = torch.from_numpy(chunk)

            speech_probability = model(
                chunk_tensor,
                SAMPLE_RATE
            ).item()

            is_speech = (
                speech_probability >= SPEECH_THRESHOLD
            )

            # ----------------------------------------
            # Speech
            # ----------------------------------------

            if is_speech:

                if not speech_started:
                    print("Speech detected...")
                    speech_started = True

                audio_chunks.append(chunk.copy())

                silence_start = None

            # ----------------------------------------
            # Silence
            # ----------------------------------------

            else:

                if speech_started:

                    audio_chunks.append(chunk.copy())

                    if silence_start is None:
                        silence_start = time.time()

                    silence_time = (
                        time.time() - silence_start
                    )

                    if silence_time >= SILENCE_DURATION:
                        print("User finished speaking.")
                        break

            # ----------------------------------------
            # Maximum utterance duration
            # ----------------------------------------

            if (
                time.time() - utterance_start_time
                >= MAX_UTTERANCE_SECONDS
            ):
                print("Maximum utterance duration reached.")
                break


    # ========================================================
    # No speech
    # ========================================================

    if not speech_started:
        print("No speech detected.")
        continue


    # ========================================================
    # Combine audio
    # ========================================================

    audio = np.concatenate(audio_chunks)

    write(
        AUDIO_FILE,
        SAMPLE_RATE,
        audio
    )


    # ========================================================
    # Whisper
    # ========================================================

    print("Calling Whisper...")

    try:

        result = subprocess.run(
            [
                str(WHISPER_CLI),
                "-m", str(WHISPER_MODEL),
                "-f", str(AUDIO_FILE),
                "-nt"
            ],
            capture_output=True,
            text=True,
            check=True
        )

    except subprocess.CalledProcessError as e:

        print("Whisper failed:")
        print(e.stderr)

        continue


    user_text = result.stdout.strip()

    if not user_text:
        print("Whisper returned empty text.")
        continue


    print("\nUser:")
    print(user_text)


    # ========================================================
    # BERT - Should AI Respond?
    # ========================================================

    inputs = tokenizer(
        user_text,
        return_tensors="np",
        truncation=True
    )

    onnx_inputs = {
        "input_ids": inputs["input_ids"],
        "attention_mask": inputs["attention_mask"]
    }

    logits = int8_session.run(
        None,
        onnx_inputs
    )[0]

    probs = softmax(logits)

    prediction = np.argmax(
        probs,
        axis=1
    )[0]

    respond_probability = probs[0][SHOULD_RESPOND_CLASS]

    print(
        f"Should respond probability: "
        f"{respond_probability:.3f}"
    )


    # ========================================================
    # Decision
    # ========================================================

    if prediction != SHOULD_RESPOND_CLASS:

        print("BERT decided: DON'T RESPOND")
        continue


    print("BERT decided: RESPOND")


    # ========================================================
    # LLM
    # ========================================================

    stream = chat(
        model=LLM_MODEL,
        messages=[
            {
                "role":"system",
                "content":"You are helpfull AI Assistent, for given text or question you will respond in short like normal person, give long response only if asked or needed"
            },
            {
                "role": "user",
                "content": user_text
            }
            ],
            stream=True
    )

    print("\nLLM:")
    for chunk in stream:
        text = chunk["message"]["content"]
        print(text, end="", flush=True)

        
    

Loaded tokenizer
Loaded INT8 BERT classification model
Listening...
Speak now.

Speech detected...
User finished speaking.
Calling Whisper...

User:
Today I have done between the interesting development by the way what is VAD.
Should respond probability: 0.670
BERT decided: RESPOND

LLM:
VAD stands for Voice Activity Detection (语音活动检测) and it is a technology used to identify speech segments in an audio signal. It's crucial for applications like voice-controlled devices, transcription software, and call center systems where accurate detection of speech signals is necessary.

Here are some key points about VAD:

1. **Purpose**: The primary goal of VAD is to determine when a human speaker is speaking within an audio stream, allowing other processing tasks to focus only on those periods where speech is present.

2. **Types**:
   - **Active Voice Detection (AVD)**: Detects the start and end times of voice segments.
   - **Silence Detection**: Identifies pauses or silence between spoken segm

KeyboardInterrupt: 

In [1]:
import subprocess
from pathlib import Path
from IPython.display import Audio, display

text = "Hello! I am your autonomous mentor. I can listen to you and respond when you need help."

output_file = Path("piper_test.wav")

subprocess.run(
    [
        "python",
        "-m",
        "piper",
        "--model",
        "en_US-lessac-medium",
        "--output_file",
        str(output_file),
    ],
    input=text,
    text=True,
    check=True
)

print("Generated:", output_file)
display(Audio(str(output_file)))

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/keerthivardhan/Desktop/ProductionProjects/autonomus-assistent/.venv/lib/python3.12/site-packages/piper/__main__.py", line 258, in <module>
    main()
  File "/home/keerthivardhan/Desktop/ProductionProjects/autonomus-assistent/.venv/lib/python3.12/site-packages/piper/__main__.py", line 143, in main
    raise ValueError(
ValueError: Unable to find voice: en_US-lessac-medium (use piper.download_voices)


CalledProcessError: Command '['python', '-m', 'piper', '--model', 'en_US-lessac-medium', '--output_file', 'piper_test.wav']' returned non-zero exit status 1.

In [2]:
import sys
print(sys.executable)

/home/keerthivardhan/Desktop/ProductionProjects/autonomus-assistent/.venv/bin/python


In [3]:
import subprocess

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "piper",
        "--help"
    ],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

usage: __main__.py [-h] -m MODEL [-c CONFIG] [-i INPUT_FILE] [-f OUTPUT_FILE]
                   [-d OUTPUT_DIR] [--output-dir-naming {timestamp,text}]
                   [--output-raw] [-s SPEAKER] [--length-scale LENGTH_SCALE]
                   [--noise-scale NOISE_SCALE] [--noise-w-scale NOISE_W_SCALE]
                   [--cuda] [--sentence-silence SENTENCE_SILENCE]
                   [--volume VOLUME] [--no-normalize] [--data-dir DATA_DIR]
                   [--debug]

options:
  -h, --help            show this help message and exit
  -m MODEL, --model MODEL
                        Path to Onnx model file
  -c CONFIG, --config CONFIG
                        Path to model config file
  -i INPUT_FILE, --input-file INPUT_FILE, --input_file INPUT_FILE
                        Paths to input text files
  -f OUTPUT_FILE, --output-file OUTPUT_FILE, --output_file OUTPUT_FILE
                        Path to output WAV file (default: stdout)
  -d OUTPUT_DIR, --output-dir OUTPUT_DIR, --outpu

In [5]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "piper.download_voices",
        "en_US-joe-medium"
    ],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)


INFO:__main__:Downloaded: en_US-joe-medium



In [6]:
import os

for root, dirs, files in os.walk("."):
    for file in files:
        if "joe" in file.lower():
            print(os.path.join(root, file))

./en_US-joe-medium.onnx
./en_US-joe-medium.onnx.json


In [7]:
import subprocess
import sys
from pathlib import Path
from IPython.display import Audio, display

text = "Hello! I am your autonomous mentor. I can listen to you and respond when you need help."

# model_path = Path("./en_US-lessac-medium.onnx")
# model_path = Path("./en_US-libritts_r-medium.onnx")
model_path = Path("./en_US-joe-medium.onnx")
output_file = Path("./piper_test.wav")

subprocess.run(
    [
        sys.executable,
        "-m",
        "piper",
        "-m",
        str(model_path),
        "-f",
        str(output_file),
    ],
    input=text,
    text=True,
    check=True
)

print("Generated:", output_file)
display(Audio(str(output_file)))

Generated: piper_test.wav


In [8]:
from kokoro import KPipeline
from IPython.display import Audio, display
import soundfile as sf

pipeline = KPipeline(lang_code="a")

text = """
Hello! I am your autonomous mentor.
If you have a question while learning, you can ask me at any time.
I will try to explain things clearly and help you understand the topic.
"""

generator = pipeline(
    text,
    voice="af_heart"
)

for i, (graphemes, phonemes, audio) in enumerate(generator):
    print("Generated:", graphemes)
    print("Phonemes:", phonemes)

    output_file = f"kokoro_test_{i}.wav"

    sf.write(
        output_file,
        audio,
        24000
    )

    display(Audio(output_file))

config.json:   0%|          | 0.00/2.35k [00:00<?, ?B/s]

/home/keerthivardhan/Desktop/ProductionProjects/autonomus-assistent/.venv/lib/python3.12/site-packages/torch/nn/modules/rnn.py:1011: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
/home/keerthivardhan/Desktop/ProductionProjects/autonomus-assistent/.venv/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


kokoro-v1_0.pth: reconstructing file:   0%|          |  0.00B /  327MB            

kokoro-v1_0.pth: downloading bytes:           |  0.00B            

Using Python 3.12.3 environment at: /home/keerthivardhan/Desktop/ProductionProjects/autonomus-assistent/.venv
Resolved 1 package in 178ms
Prepared 1 package in 720ms
Installed 1 package in 5ms
 + en-core-web-sm==3.8.0 (from https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl)
/home/keerthivardhan/Desktop/ProductionProjects/autonomus-assistent/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


...

voices/af_heart.pt: reconstructing file:   0%|          |  0.00B /  523kB            

voices/af_heart.pt: downloading bytes:           |  0.00B            

Generated: Hello! I am your autonomous mentor.
Phonemes: həlˈO! ˌI ɐm jʊɹ ɔtˈɑnəməs mˈɛntˌɔɹ.


Generated: If you have a question while learning, you can ask me at any time.
Phonemes: ˌɪf ju hæv ɐ kwˈɛsʧᵊn wˌIl lˈɜɹnɪŋ, ju kæn ˈæsk mˌi æt ˈɛni tˈIm.


Generated: I will try to explain things clearly and help you understand the topic.
Phonemes: ˌI wɪl tɹˈI tʊ ɪksplˈAn θˈɪŋz klˈɪɹli ænd hˈɛlp ju ˌʌndəɹstˈænd ðə tˈɑpɪk.


In [9]:
from kokoro import KPipeline
from IPython.display import Audio, display
import soundfile as sf
import time

text = """
Hello! I am your autonomous mentor.
If you have a question while learning, you can ask me at any time.
I will explain things clearly and help you understand the topic.
"""

pipeline = KPipeline(lang_code="a")

voices = [
    "af_heart",
    "af_bella",
    "am_michael",
    "bm_george",
]

for voice in voices:
    print(f"\n===== {voice} =====")

    start = time.perf_counter()

    generator = pipeline(
        text,
        voice=voice
    )

    audio_parts = []

    for graphemes, phonemes, audio in generator:
        audio_parts.append(audio)

    audio = audio_parts[0] if len(audio_parts) == 1 else __import__("numpy").concatenate(audio_parts)

    elapsed = time.perf_counter() - start

    output_file = f"kokoro_{voice}.wav"

    sf.write(
        output_file,
        audio,
        24000
    )

    duration = len(audio) / 24000

    print(f"Generation time : {elapsed:.2f} sec")
    print(f"Audio duration   : {duration:.2f} sec")
    print(f"Realtime factor  : {elapsed / duration:.2f}x")

    display(Audio(output_file))

/home/keerthivardhan/Desktop/ProductionProjects/autonomus-assistent/.venv/lib/python3.12/site-packages/torch/nn/modules/rnn.py:1011: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
/home/keerthivardhan/Desktop/ProductionProjects/autonomus-assistent/.venv/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)



===== af_heart =====
Generation time : 4.92 sec
Audio duration   : 10.88 sec
Realtime factor  : 0.45x



===== af_bella =====


voices/af_bella.pt: reconstructing file:   0%|          |  0.00B /  523kB            

voices/af_bella.pt: downloading bytes:           |  0.00B            

...

Generation time : 7.57 sec
Audio duration   : 12.62 sec
Realtime factor  : 0.60x



===== bm_george =====


voices/bm_george.pt: reconstructing file:   0%|          |  0.00B /  523kB            

voices/bm_george.pt: downloading bytes:           |  0.00B            

Generation time : 7.78 sec
Audio duration   : 12.68 sec
Realtime factor  : 0.61x


In [ ]:
from kokoro import KPipeline
from IPython.display import Audio, display
import soundfile as sf
import time

text = """
Hello! I am your autonomous mentor.
If you have a question while learning, you can ask me at any time.
I will explain things clearly and help you understand the topic.
"""

pipeline = KPipeline(lang_code="a")

voice = "af_heart"

print(f"\n===== {voice} =====")

start = time.perf_counter()

generator = pipeline(
    text,
    voice=voice
)

audio_parts = []

for graphemes, phonemes, audio in generator:
    audio_parts.append(audio)

audio = audio_parts[0] if len(audio_parts) == 1 else __import__("numpy").concatenate(audio_parts)

elapsed = time.perf_counter() - start

output_file = f"kokoro_{voice}.wav"

sf.write(
    output_file,
    audio,
    24000
)

duration = len(audio) / 24000

print(f"Generation time : {elapsed:.2f} sec")
print(f"Audio duration   : {duration:.2f} sec")
print(f"Realtime factor  : {elapsed / duration:.2f}x")

display(Audio(output_file))